# NB02 — Trace Generation

**Responsibility:** Run PyTorch profiler traces and save Chrome trace JSON files to `TRACE_DIR`.  
**Output:** `./traces/trace_<config>_<id>.json` files (501 total across 3 configs)  
**Next step:** Run NB04 to build DAGs and convert to `graphs_v2.pkl`


In [1]:
!pip install torch torchvision networkx tqdm matplotlib

In [2]:
import os

DRIVE_MOUNT_POINT = "/content/drive"
DRIVE_FOLDER = "MyDrive/BottleneckOracle_Backup2"
IN_COLAB = False

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT_POINT)
    IN_COLAB = True
except Exception:
    print("Colab Drive not available; using local workspace paths.")

DRIVE_BASE = os.path.join(DRIVE_MOUNT_POINT, DRIVE_FOLDER) if IN_COLAB else "."
TRACE_DIR  = os.path.join(DRIVE_BASE, "traces")

os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(TRACE_DIR, exist_ok=True)

print(f"Trace output dir: {TRACE_DIR}")


Mounted at /content/drive
Trace output dir: /content/drive/MyDrive/BottleneckOracle_Backup2/traces


In [3]:
configs = [
    {"name": "tiny",   "d_model": 64,  "nhead": 2, "num_layers": 1},
    {"name": "small",  "d_model": 128, "nhead": 4, "num_layers": 2},
    {"name": "medium", "d_model": 256, "nhead": 8, "num_layers": 4},
]

print("Configs ready.")

Configs ready.


In [4]:
import json
import numpy as np
import torch
import torch.nn as nn
from torch.profiler import profile, ProfilerActivity
from tqdm import tqdm
import warnings
import shutil

warnings.filterwarnings("ignore")

print("Libraries loaded.")


Libraries loaded.


In [5]:
TARGET_TRACES_PER_CONFIG = 167

print(f"Will generate up to {TARGET_TRACES_PER_CONFIG} traces per config ({TARGET_TRACES_PER_CONFIG * 3} total).")


Will generate up to 167 traces per config (501 total).


## Trace Generator

Runs two concurrent Transformer forward passes per trace using threading.  
This creates genuine overlapping ops → real fork-join structure for the DAG builder.  
Sequence length and batch size are varied per `trace_id` for diversity.


In [6]:
def generate_trace(config, trace_id):
    """
    Generate a trace with realistic parallelism by running multiple
    independent forward passes concurrently using threading.
    This creates genuine concurrent ops in the trace, giving the DAG
    builder real fork-join structure to work with.
    """
    import threading

    # Use 2-3 model variants to simulate pipeline stages
    models = [
        nn.Transformer(
            d_model=config["d_model"],
            nhead=config["nhead"],
            num_encoder_layers=config["num_layers"],
            num_decoder_layers=config["num_layers"]
        )
        for _ in range(2)  # two concurrent "pipeline stages"
    ]

    # Vary sequence length and batch size per trace_id for diversity
    rng = np.random.default_rng(trace_id)
    seq_len = int(rng.integers(4, 16))
    batch   = int(rng.integers(8, 48))

    inputs = [
        (torch.rand(seq_len, batch, config["d_model"]),
         torch.rand(seq_len, batch, config["d_model"]))
        for _ in models
    ]

    filepath = os.path.join(TRACE_DIR, f"trace_{config['name']}_{trace_id}.json")

    with profile(
        activities=[ProfilerActivity.CPU],
        record_shapes=True,
        with_stack=True,
    ) as prof:
        threads = []
        for model, (src, tgt) in zip(models, inputs):
            t = threading.Thread(target=lambda m=model, s=src, g=tgt: m(s, g))
            threads.append(t)
        # stagger starts slightly to create dependency diversity
        for i, t in enumerate(threads):
            t.start()
            if i < len(threads) - 1:
                import time; time.sleep(0.001 * int(rng.integers(1, 5)))
        for t in threads:
            t.join()

    prof.export_chrome_trace(filepath)
    return filepath

## Run — Generate All Traces

In [7]:
def generate_all_traces():
    """Generate raw trace JSON files for every config/trace id."""
    total = 0
    skipped = 0
    target_total = len(configs) * 167
    print(f"Generating {target_total} traces into {TRACE_DIR} ...\n")

    for cfg in configs:
        print(f"── Config: {cfg['name']} ──────────────────────────────")
        for trace_id in range(167):
            save_path = os.path.join(TRACE_DIR, f"trace_{cfg['name']}_{trace_id}.json")
            if os.path.exists(save_path):
                skipped += 1
                continue

            try:
                trace_path = generate_trace(cfg, trace_id)
                total += 1
                if (total + skipped) % 50 == 0:
                    print(f"  Progress: {total + skipped} / {target_total} ({skipped} skipped)")
            except Exception as exc:
                print(f"  [SKIP] trace_id={trace_id} — {exc}")

    print(f"\nDone. Generated {total} new traces, skipped {skipped} existing.")


print("Trace generation loop defined.")

Trace generation loop defined.


In [8]:
os.makedirs(TRACE_DIR, exist_ok=True)
generate_all_traces()


Generating 501 traces into /content/drive/MyDrive/BottleneckOracle_Backup2/traces ...

── Config: tiny ──────────────────────────────
  Progress: 50 / 501 (0 skipped)
  Progress: 100 / 501 (0 skipped)
  Progress: 150 / 501 (0 skipped)
── Config: small ──────────────────────────────
  Progress: 200 / 501 (0 skipped)
  Progress: 250 / 501 (0 skipped)
  Progress: 300 / 501 (0 skipped)
── Config: medium ──────────────────────────────
  Progress: 350 / 501 (0 skipped)
  Progress: 400 / 501 (0 skipped)
  Progress: 450 / 501 (0 skipped)
  Progress: 500 / 501 (0 skipped)

Done. Generated 501 new traces, skipped 0 existing.


## Backup Traces to Drive

Zip and upload the generated trace JSONs so NB04 can restore them in a fresh Colab session.


In [9]:
if IN_COLAB:
    BACKUP_ZIP = os.path.join(DRIVE_BASE, "traces_backup")
    shutil.make_archive(BACKUP_ZIP, "zip", TRACE_DIR)
    print(f"✅ Traces zipped to: {BACKUP_ZIP}.zip")
else:
    print("Skipped Drive backup (not in Colab). Traces are in:", TRACE_DIR)


✅ Traces zipped to: /content/drive/MyDrive/BottleneckOracle_Backup2/traces_backup.zip
